In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os, sys

GOOGLE_DRIVE_PATH_POST_MYDRIVE = 'toxic_comment_detection'
GOOGLE_DRIVE_PATH = os.path.join('/content', 'drive', 'MyDrive', GOOGLE_DRIVE_PATH_POST_MYDRIVE)
print(os.listdir(GOOGLE_DRIVE_PATH))

os.chdir(GOOGLE_DRIVE_PATH)
sys.path.append(GOOGLE_DRIVE_PATH)

['.env', 'kaggle.json', '.pre-commit-config.yaml', '.python-version', '.gitignore', 'README.md', 'train.py', 'dataset_analysis.py', 'uv.lock', 'pyproject.toml', '.git', '.venv', 'notebooks', 'models', 'docs', 'dataset', '.ruff_cache', 'model_output']


In [ ]:
# if running locally set GOOGLE PATH
import sys
if 'google.colab' in sys.modules:
  print(f'Running in google colab. Our path is `{GOOGLE_DRIVE_PATH}`')
else:
  GOOGLE_DRIVE_PATH = '.'
  print('Running locally.')

Running in google colab. Our path is `/content/drive/MyDrive/toxic_comment_detection`


In [ ]:
!pip install transformers accelerate peft bitsandbytes datasets trl torch torchvision torchaudio

In [ ]:
import torch
import numpy as np
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments
)
from sklearn.metrics import classification_report, f1_score


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


In [ ]:
class Args:
    model_name = "meta-llama/Llama-3.2-1B-Instruct"
    batch_size = 32
    eval_batch_size = 32
    learning_rate = 2e-5
    weight_decay = 0.01
    epochs = 3
    grad_accum_steps = 1
    fp16 = True
    gradient_checkpoint = True
    optimizer = "adamw"
    warmup_ratio = 0.1
    max_length = 256
    val_ratio = 0.2
    test_ratio = 0.2
    output_dir = "./model_output"
    use_class_weights = True

args = Args()

In [ ]:
data = load_dataset("thesofakillers/jigsaw-toxic-comment-classification-challenge")
full_train = data["train"]

test_val_split = full_train.train_test_split(test_size=args.test_ratio, seed=42)
train_temp = test_val_split["train"]
test_set = test_val_split["test"]

val_fraction = args.val_ratio / (1 - args.test_ratio)
train_val_split = train_temp.train_test_split(test_size=val_fraction, seed=42)
train_set = train_val_split["train"]
val_set = train_val_split["test"]

print(f"Split sizes: {len(train_set)} train, {len(val_set)} validation, {len(test_set)} test")


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/1.43k [00:00<?, ?B/s]

train.csv:   0%|          | 0.00/68.8M [00:00<?, ?B/s]

test.csv:   0%|          | 0.00/60.4M [00:00<?, ?B/s]

test_labels.csv:   0%|          | 0.00/4.98M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/159571 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/306328 [00:00<?, ? examples/s]

Split sizes: 95742 train, 31914 validation, 31915 test


In [ ]:
# Cell 4: Tokenizer and data encoding
tokenizer = AutoTokenizer.from_pretrained(args.model_name, use_fast=True)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token_id = tokenizer.eos_token_id
    tokenizer.padding_side = "right"

def preprocess_batch(batch):
    encodings = tokenizer(batch["comment_text"], truncation=True, max_length=args.max_length, padding="longest")
    labels = np.array([batch[label] for label in ["toxic", "severe_toxic", "obscene", "threat", "insult", "identity_hate"]]).T
    encodings["labels"] = labels.tolist()
    return encodings

cols_to_remove = ["id", "comment_text", "toxic", "severe_toxic", "obscene", "threat", "insult", "identity_hate"]
train_enc = train_set.map(preprocess_batch, batched=True, remove_columns=cols_to_remove)
val_enc = val_set.map(preprocess_batch, batched=True, remove_columns=cols_to_remove)
test_enc = test_set.map(preprocess_batch, batched=True, remove_columns=cols_to_remove)


In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    args.model_name,
    num_labels=6,
    problem_type="multi_label_classification"
).to(device)

model.config.pad_token_id = tokenizer.pad_token_id

if args.gradient_checkpoint:
    model.gradient_checkpointing_enable()

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

pos_weight = None
if args.use_class_weights:
    train_labels = np.array(train_enc["labels"])
    pos_counts = train_labels.sum(axis=0)
    neg_counts = len(train_enc) - pos_counts
    pos_weight_values = neg_counts / np.clip(pos_counts, 1, None)
    pos_weight = torch.tensor(pos_weight_values, dtype=torch.float).to(device)

# class MultilabelTrainer(Trainer):
#     def compute_loss(self, model, inputs, return_outputs=False,num_items_in_batch=None):
#         labels = inputs.pop("labels")  # Remove labels from inputs dictionary
#         outputs = model(**inputs)      # Forward pass without labels
#         logits = outputs.get("logits")
#         # Ensure labels are float32
#         labels = labels.to(logits.dtype)
#         loss_fct = torch.nn.BCEWithLogitsLoss(pos_weight=pos_weight)
#         loss = loss_fct(logits, labels)
#         return (loss, outputs) if return_outputs else loss

class MultilabelTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.get("logits")
        labels = labels.to(logits.dtype)

        # Focal Loss implementation
        gamma = 2.0
        alpha = 0.25

        bce_loss = torch.nn.functional.binary_cross_entropy_with_logits(logits, labels, reduction="none")
        probas = torch.sigmoid(logits)
        pt = torch.where(labels == 1, probas, 1 - probas)
        focal_loss = (alpha * (1 - pt) ** gamma * bce_loss).mean()

        return (focal_loss, outputs) if return_outputs else focal_loss

trainer = MultilabelTrainer(
    model=model,
    args=TrainingArguments(
        output_dir=args.output_dir,
        num_train_epochs=args.epochs,
        per_device_train_batch_size=args.batch_size,
        per_device_eval_batch_size=args.eval_batch_size,
        gradient_accumulation_steps=args.grad_accum_steps,
        learning_rate=args.learning_rate,
        weight_decay=args.weight_decay,
        fp16=args.fp16,
        eval_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="f1_macro",
        greater_is_better=True,
        logging_strategy="epoch",
        report_to="none",
        warmup_ratio=args.warmup_ratio,
        optim="adamw_torch" if args.optimizer == "adamw" else "adafactor"
    ),
    train_dataset=train_enc,
    eval_dataset=val_enc,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=lambda eval_pred: {
        "f1_macro": f1_score(eval_pred.label_ids, (torch.sigmoid(torch.tensor(eval_pred.predictions)) >= 0.5).int(), average="macro"),
        "f1_micro": f1_score(eval_pred.label_ids, (torch.sigmoid(torch.tensor(eval_pred.predictions)) >= 0.5).int(), average="micro")
    }
)


Some weights of LlamaForSequenceClassification were not initialized from the model checkpoint at meta-llama/Llama-3.2-1B-Instruct and are newly initialized: ['score.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
<ipython-input-15-46c8563abcdb>:51: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `MultilabelTrainer.__init__`. Use `processing_class` instead.
  trainer = MultilabelTrainer(


In [ ]:
trainer.train()



Epoch,Training Loss,Validation Loss,F1 Macro,F1 Micro
1,0.004800,0.002589,0.624053,0.769137
2,0.002100,0.002612,0.623539,0.785576
3,0.001000,0.003453,0.647903,0.786563


TrainOutput(global_step=8976, training_loss=0.0026127360561952234, metrics={'train_runtime': 6700.7799, 'train_samples_per_second': 42.865, 'train_steps_per_second': 1.34, 'total_flos': 4.293371821031424e+17, 'train_loss': 0.0026127360561952234, 'epoch': 3.0})

In [ ]:

predictions = trainer.predict(test_enc)
y_true = predictions.label_ids
y_prob = torch.sigmoid(torch.tensor(predictions.predictions)).numpy()
y_pred = (y_prob >= 0.5).astype(int)

target_names = ["toxic", "severe_toxic", "obscene", "threat", "insult", "identity_hate"]
print(classification_report(y_true, y_pred, target_names=target_names, digits=4))


               precision    recall  f1-score   support

        toxic     0.7676    0.8764    0.8184      3018
 severe_toxic     0.3983    0.6288    0.4877       299
      obscene     0.7449    0.9075    0.8182      1676
       threat     0.4862    0.6092    0.5408        87
       insult     0.6920    0.8529    0.7641      1570
identity_hate     0.4927    0.6306    0.5532       268

    micro avg     0.7086    0.8550    0.7750      6918
    macro avg     0.5969    0.7509    0.6637      6918
 weighted avg     0.7148    0.8550    0.7779      6918
  samples avg     0.0701    0.0806    0.0723      6918



/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in samples with no predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in samples with no true labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 in samples with no true nor predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [ ]:

predictions = trainer.predict(test_enc)
y_true = predictions.label_ids
y_prob = torch.sigmoid(torch.tensor(predictions.predictions)).numpy()
y_pred = (y_prob >= 0.5).astype(int)

target_names = ["toxic", "severe_toxic", "obscene", "threat", "insult", "identity_hate"]
print(classification_report(y_true, y_pred, target_names=target_names, digits=4))


               precision    recall  f1-score   support

        toxic     0.8138    0.8429    0.8281      3018
 severe_toxic     0.5045    0.3779    0.4321       299
      obscene     0.8402    0.8126    0.8262      1676
       threat     0.7222    0.4483    0.5532        87
       insult     0.7768    0.7471    0.7617      1570
identity_hate     0.6354    0.4552    0.5304       268

    micro avg     0.7957    0.7738    0.7846      6918
    macro avg     0.7155    0.6140    0.6553      6918
 weighted avg     0.7904    0.7738    0.7805      6918
  samples avg     0.0738    0.0726    0.0707      6918



/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in samples with no predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in samples with no true labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 in samples with no true nor predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


## Running after training ( focal loss)

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer


checkpoint_path = "./model_output/checkpoint-8976"
model = AutoModelForSequenceClassification.from_pretrained(checkpoint_path)
tokenizer = AutoTokenizer.from_pretrained(checkpoint_path)

trainer = Trainer(model=model, tokenizer=tokenizer)


<ipython-input-9-2c6f9bfc286f>:8: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(model=model, tokenizer=tokenizer)


In [ ]:
class MultilabelTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.get("logits")
        labels = labels.to(logits.dtype)

        # Focal Loss implementation
        gamma = 2.0
        alpha = 0.25

        bce_loss = torch.nn.functional.binary_cross_entropy_with_logits(logits, labels, reduction="none")
        probas = torch.sigmoid(logits)
        pt = torch.where(labels == 1, probas, 1 - probas)
        focal_loss = (alpha * (1 - pt) ** gamma * bce_loss).mean()

        return (focal_loss, outputs) if return_outputs else focal_loss

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

trainer = MultilabelTrainer(
    model=model,
    args=TrainingArguments(
        output_dir=args.output_dir,
        num_train_epochs=args.epochs,
        per_device_train_batch_size=args.batch_size,
        per_device_eval_batch_size=args.eval_batch_size,
        gradient_accumulation_steps=args.grad_accum_steps,
        learning_rate=args.learning_rate,
        weight_decay=args.weight_decay,
        fp16=args.fp16,
        eval_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="f1_macro",
        greater_is_better=True,
        logging_strategy="epoch",
        report_to="none",
        warmup_ratio=args.warmup_ratio,
        optim="adamw_torch" if args.optimizer == "adamw" else "adafactor"
    ),
    train_dataset=train_enc,
    eval_dataset=val_enc,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=lambda eval_pred: {
        "f1_macro": f1_score(eval_pred.label_ids, (torch.sigmoid(torch.tensor(eval_pred.predictions)) >= 0.5).int(), average="macro"),
        "f1_micro": f1_score(eval_pred.label_ids, (torch.sigmoid(torch.tensor(eval_pred.predictions)) >= 0.5).int(), average="micro")
    }
)

<ipython-input-18-29988824d0fd>:21: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `MultilabelTrainer.__init__`. Use `processing_class` instead.
  trainer = MultilabelTrainer(


In [ ]:
if tokenizer.pad_token_id is None:
    tokenizer.pad_token_id = tokenizer.eos_token_id
    tokenizer.padding_side = "right"

def preprocess_batch(batch):
    encodings = tokenizer(batch["comment_text"], truncation=True, max_length=args.max_length, padding="longest")
    labels = np.array([batch[label] for label in ["toxic", "severe_toxic", "obscene", "threat", "insult", "identity_hate"]]).T
    encodings["labels"] = labels.tolist()
    return encodings

cols_to_remove = ["id", "comment_text", "toxic", "severe_toxic", "obscene", "threat", "insult", "identity_hate"]
train_enc = train_set.map(preprocess_batch, batched=True, remove_columns=cols_to_remove)
val_enc = val_set.map(preprocess_batch, batched=True, remove_columns=cols_to_remove)
test_enc = test_set.map(preprocess_batch, batched=True, remove_columns=cols_to_remove)

Map:   0%|          | 0/95742 [00:00<?, ? examples/s]

Map:   0%|          | 0/31914 [00:00<?, ? examples/s]

Map:   0%|          | 0/31915 [00:00<?, ? examples/s]

In [ ]:
predictions = trainer.predict(test_enc)

y_true = predictions.label_ids
y_prob = torch.sigmoid(torch.tensor(predictions.predictions)).numpy()


In [ ]:
from sklearn.metrics import roc_auc_score

# Compute AUC
macro_auc = roc_auc_score(y_true, y_prob, average='macro')
micro_auc = roc_auc_score(y_true, y_prob, average='micro')

print(f"Macro AUC: {macro_auc:.4f}")
print(f"Micro AUC: {micro_auc:.4f}")


Macro AUC: 0.9902
Micro AUC: 0.9908


In [ ]:
from sklearn.metrics import roc_auc_score

# Compute AUC
macro_auc = roc_auc_score(y_true, y_prob, average='macro')
micro_auc = roc_auc_score(y_true, y_prob, average='micro')

print(f"Macro AUC: {macro_auc:.4f}")
print(f"Micro AUC: {micro_auc:.4f}")


Macro AUC: 0.9902
Micro AUC: 0.9908
